# Sensitivity Analysis and Figure Generation
**Simulation-Informed Deep Learning for Estimating Material Parameters from Microstructural Morphology**

This notebook generates all analysis figures for the paper:
- Model performance (layer sweep, loss, parity plots)
- Morphology-based interpretability (interface density, domain size, error–similarity)
- preprocessing: STEM experimental image pipeline

Run cells top to bottom. The only cells you need to edit are:
- **Cell 2** — file paths and physical constants
- **Cell 7** — exclusion threshold (if tuning)
- **Cell 10** — model performance figure paths (separate dataset)
- **Cell 12** — interpolation/extended grid figure paths
- **Cell 14** — STEM image path

---

## Imports
Standard libraries for image processing, morphology analysis, and plotting.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="findfont")

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from skimage import filters, morphology, measure, segmentation
from scipy import ndimage as ndi

## Paths and Physical Constants
 **This is the only cell you need to edit when switching datasets.**

- `DOMAIN_SIZE_NM`: fixed physical domain size for all synthetic images
- `MIN_AREA`: minimum pixel area to count as a real Cr-rich domain (noise filter)
- `BORDER_THRESHOLD`: phase fraction cutoff for excluding non-droplet morphologies

In [ ]:
# ----- Morphology analysis inputs -----
CSV_PATH  = "/path/to/images/data.csv"  # CSV must have columns: image_name, kappa, mob


IMAGE_DIR  = "/path/to/images"  # directory containing all image files 

# ----- Output directory for figures -----
OUT_DIR    = "/path/to/output/figures" # directory to save generated figures

os.makedirs(OUT_DIR, exist_ok=True)

# ----- Physical constants -----
DOMAIN_SIZE_NM   = 50.0   # all synthetic images represent a 50 nm x 50 nm domain
MIN_AREA         = 20     # pixels; well below real domain size (~1000 px), just removes noise specks
BORDER_THRESHOLD = 0.35   # phase fraction cutoff for exclusion (see Cell 7)

## Load CSV and Verify Images
Loads the prediction errors CSV and spot-checks that the first 5 images exist on disk.
If any show `MISSING`, check that `IMAGE_DIR` in Cell 2 points to the right folder.

In [ ]:
#df = pd.read_csv(CSV_PATH)
df = pd.read_csv(CSV_PATH, header=None, names=["image_name", "kappa_gt", "mob_gt"])

pred_df = pd.read_csv("/home/asfandyarkhan/Desktop/Papers/CNN_Paper/Our_Model/Results/760_set_layer464/prediction_errors.csv")

df = df.merge(
    pred_df[["image_name", "kappa_pred", "mob_pred", "err_kappa", "err_mob", "err_total"]],
    on="image_name",
    how="left"   
)

print(f"Total images:        {len(df)}")
print(f"With predictions:    {df['err_total'].notna().sum()}")
print(f"Without predictions: {df['err_total'].isna().sum()}")

print(f"Total rows in CSV: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print()

print("Spot-check first 5 image paths:")
for name in df["image_name"].head():
    path = os.path.join(IMAGE_DIR, name)
    print(f"  {name}  ->  {'OK' if os.path.exists(path) else 'MISSING'}")

## Morphology Extraction Function
Defines `extract_morphology_metrics()` — called once per image in Cell 6.

Key design decisions:
- `cr_dark=True`: Cr-rich phase is dark in these COMSOL images
- `binary_fill_holes` runs **before** `remove_small_objects` (correct order — fill interior voids first)
- All metrics returned in physical units (nm, nm⁻¹), not pixels
- `mode='inner'` for boundary detection counts pixels on the object side only

In [ ]:
def extract_morphology_metrics(image_path, domain_size_nm=50.0, min_area=20, cr_dark=True):

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(image_path)

    img_norm = img.astype(np.float32) / 255.0
    height_px, width_px = img.shape

    nm_per_pixel_x   = domain_size_nm / width_px
    nm_per_pixel_y   = domain_size_nm / height_px
    pixel_area_nm2   = nm_per_pixel_x * nm_per_pixel_y
    nm_per_pixel_eff = np.sqrt(pixel_area_nm2)

    # Otsu threshold — automatically finds best dark/bright split
    threshold  = filters.threshold_otsu(img_norm)
    phase_mask = (img_norm < threshold) if cr_dark else (img_norm > threshold)

    # fill holes first, then remove small objects
    phase_mask = ndi.binary_fill_holes(phase_mask)
    phase_mask = morphology.remove_small_objects(phase_mask, min_size=min_area)

    # connected domain analysis
    labels = measure.label(phase_mask)
    props  = measure.regionprops(labels)

    diameters, areas = [], []
    for p in props:
        if p.area >= min_area:
            diameters.append(p.equivalent_diameter)
            areas.append(p.area)

    if len(diameters) > 0:
        avg_domain_size_px    = np.mean(diameters)
        median_domain_size_px = np.median(diameters)
        max_domain_size_px    = np.max(diameters)
        avg_domain_area_px    = np.mean(areas)
        num_domains           = len(diameters)
    else:
        avg_domain_size_px = median_domain_size_px = max_domain_size_px = avg_domain_area_px = np.nan
        num_domains = 0

    # convert pixel measurements to nm
    avg_domain_size_nm    = avg_domain_size_px    * nm_per_pixel_eff
    median_domain_size_nm = median_domain_size_px * nm_per_pixel_eff
    max_domain_size_nm    = max_domain_size_px    * nm_per_pixel_eff
    avg_domain_area_nm2   = avg_domain_area_px    * pixel_area_nm2

    # interface density: total boundary length / image area
    boundaries               = segmentation.find_boundaries(phase_mask, mode="inner")
    interface_length_nm      = boundaries.sum() * nm_per_pixel_eff
    image_area_nm2           = domain_size_nm * domain_size_nm
    interface_density_nm_inv = interface_length_nm / image_area_nm2

    # image gradient (used as a decomposition signal strength check)
    gx       = cv2.Sobel(img_norm, cv2.CV_32F, 1, 0)
    gy       = cv2.Sobel(img_norm, cv2.CV_32F, 0, 1)
    grad_mag = np.sqrt(gx**2 + gy**2)

    return {
        "image_width_px":           width_px,
        "image_height_px":          height_px,
        "nm_per_pixel_x":           nm_per_pixel_x,
        "nm_per_pixel_y":           nm_per_pixel_y,
        "nm_per_pixel_eff":         nm_per_pixel_eff,
        "avg_domain_size_px":       avg_domain_size_px,
        "median_domain_size_px":    median_domain_size_px,
        "max_domain_size_px":       max_domain_size_px,
        "avg_domain_size_nm":       avg_domain_size_nm,
        "median_domain_size_nm":    median_domain_size_nm,
        "max_domain_size_nm":       max_domain_size_nm,
        "avg_domain_area_nm2":      avg_domain_area_nm2,
        "num_domains":              num_domains,
        "interface_density_px":     boundaries.sum() / phase_mask.size,
        "interface_density_nm_inv": interface_density_nm_inv,
        "avg_gradient":             np.mean(grad_mag),
        "median_gradient":          np.median(grad_mag),
        "max_gradient":             np.max(grad_mag),
        "phase_fraction":           np.mean(phase_mask),
        "otsu_threshold":           threshold,
    }

## Single Image Diagnostic
Visual check for one image. Change `idx` to inspect any row from the CSV.

Shows: original | detected Cr-rich mask | labeled domains | boundary overlay

Prints physical metrics below the plot.

> This cell is for inspection only — does not affect downstream results.

In [ ]:
idx        = 20   # change this to inspect any image
image_name = df.iloc[idx]["image_name"]
image_path = os.path.join(IMAGE_DIR, image_name)

print(f"Index:     {idx}")
print(f"Image:     {image_name}")
print(f"kappa_gt:  {df.iloc[idx]['kappa_gt']}")
print(f"mob_gt:    {df.iloc[idx]['mob_gt']}")
print("-" * 40)

img      = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
img_norm = img.astype(np.float32) / 255.0
h, w     = img.shape

nm_per_pixel_x   = DOMAIN_SIZE_NM / w
nm_per_pixel_y   = DOMAIN_SIZE_NM / h
pixel_area_nm2   = nm_per_pixel_x * nm_per_pixel_y
nm_per_pixel_eff = np.sqrt(pixel_area_nm2)

thr  = filters.threshold_otsu(img_norm)
mask = img_norm < thr
mask = ndi.binary_fill_holes(mask)
mask = morphology.remove_small_objects(mask, min_size=MIN_AREA)

labels = measure.label(mask)
props  = measure.regionprops(labels)
diam   = [p.equivalent_diameter for p in props if p.area >= MIN_AREA]

bounds  = segmentation.find_boundaries(mask, mode="inner")
overlay = np.dstack([img_norm, img_norm, img_norm])
overlay[bounds] = [1, 0, 0]

fig, ax = plt.subplots(2, 2, figsize=(12, 10))
ax[0,0].imshow(img,    cmap="gray");          ax[0,0].set_title("Original");               ax[0,0].axis("off")
ax[0,1].imshow(mask,   cmap="gray");          ax[0,1].set_title("Detected Cr-rich phase");  ax[0,1].axis("off")
ax[1,0].imshow(labels, cmap="nipy_spectral"); ax[1,0].set_title(f"Domains = {len(diam)}"); ax[1,0].axis("off")
ax[1,1].imshow(overlay);                      ax[1,1].set_title("Detected interfaces");     ax[1,1].axis("off")
plt.tight_layout()
plt.show()

avg_d = np.mean(diam)
max_d = np.max(diam)
id_nm = (bounds.sum() * nm_per_pixel_eff) / (DOMAIN_SIZE_NM ** 2)

print(f"\nMetrics")
print(f"----------")
print(f"Image name:            {image_name}")
print(f"Image size:            {w} x {h} pixels")
print(f"nm/pixel:              {nm_per_pixel_eff:.4f}")
print(f"Cr phase fraction:     {np.mean(mask):.4f}")
print(f"Number of domains:     {len(diam)}")
print(f"Average domain diam:   {avg_d:.2f} px = {avg_d*nm_per_pixel_eff:.2f} nm")
print(f"Maximum domain diam:   {max_d:.2f} px = {max_d*nm_per_pixel_eff:.2f} nm")
print(f"Interface density:     {id_nm:.5f} nm^-1")

## Extract Metrics for All Images
Runs `extract_morphology_metrics()` on all images and merges results with the CSV.
Saves full metrics to `test_set_morphology_metrics.csv` in `OUT_DIR`.

If any images are missing, check `IMAGE_DIR` in Cell 2.

In [ ]:
all_metrics = []

for _, row in df.iterrows():
    image_path = os.path.join(IMAGE_DIR, row["image_name"])
    try:
        metrics = extract_morphology_metrics(
            image_path=image_path,
            domain_size_nm=DOMAIN_SIZE_NM,
            min_area=MIN_AREA,
            cr_dark=True
        )
        metrics["image_found"] = True
    except Exception as e:
        print(f"ERROR: {row['image_name']} — {e}")
        metrics = {"image_found": False}
    all_metrics.append(metrics)

metrics_df = pd.DataFrame(all_metrics)
out_df     = pd.concat([df.reset_index(drop=True), metrics_df], axis=1)

print(f"Total images:  {len(out_df)}")
print(f"Images found:  {out_df['image_found'].sum()}")
display(out_df.head())

metrics_csv = os.path.join(OUT_DIR, "test_set_morphology_metrics.csv")
out_df.to_csv(metrics_csv, index=False)
print(f"Saved: {metrics_csv}")

## Exclusion with Visual Confirmation
 **Only change `BORDER_THRESHOLD` in Cell 2 — everything here auto-updates.**

Three exclusion rules applied to `out_df`:

| Rule | Condition | Reason |
|---|---|---|
| Bicontinuous | `phase_fraction > 0.5` | Otsu thresholded the wrong phase |
| Borderline blurry | `phase_fraction >= BORDER_THRESHOLD` | Partial/incomplete decomposition |
| No decomposition | `phase_fraction < 0.01` | Image is essentially uniform gray |

Output: `out_df_clean` — used in all downstream cells.
Visual confirmation grid saved to `excluded_images_confirmation.png`.

In [ ]:
n_before = len(out_df)

mask_bicont   = out_df["phase_fraction"] > 0.5
mask_border   = out_df["phase_fraction"] >= BORDER_THRESHOLD
mask_nodecomp = out_df["phase_fraction"] < 0.01

out_df_clean = out_df[
    ~mask_border & ~mask_nodecomp
].copy().reset_index(drop=True)

excluded_df = out_df[mask_border | mask_nodecomp].copy().reset_index(drop=True)

def exclusion_reason(pf):
    if pf > 0.5:             return "bicontinuous (pf > 0.5)"
    if pf >= BORDER_THRESHOLD: return f"borderline blurry (pf >= {BORDER_THRESHOLD})"
    return "no decomposition (pf < 0.01)"

excluded_df["reason"] = excluded_df["phase_fraction"].apply(exclusion_reason)
border_label = f"borderline blurry (pf >= {BORDER_THRESHOLD})"

reason_colors = {
    "bicontinuous (pf > 0.5)":      "red",
    border_label:                    "darkorange",
    "no decomposition (pf < 0.01)": "steelblue",
}

print(f"BORDER_THRESHOLD:                      {BORDER_THRESHOLD}")
print(f"Total original:                        {n_before}")
print(f"Excluded bicontinuous  (pf > 0.50):    {mask_bicont.sum()}")
print(f"Excluded borderline    (pf >= {BORDER_THRESHOLD}):  {(mask_border & ~mask_bicont).sum()}")
print(f"Excluded no-decomp     (pf < 0.01):    {mask_nodecomp.sum()}")
print(f"Remaining clean:                       {len(out_df_clean)}")
print()
print("Excluded images:")
print(excluded_df[[
    "image_name", "kappa_gt", "mob_gt",
    "phase_fraction", "avg_domain_size_nm",
    "num_domains", "reason"
]].to_string(index=False))

# visual confirmation grid
n   = len(excluded_df)
fig, axes = plt.subplots(n, 4, figsize=(15, n * 3.0))
if n == 1:
    axes = axes[np.newaxis, :]

fig.suptitle(
    f"Excluded Images (n={n})  |  border threshold = {BORDER_THRESHOLD}",
    fontsize=13, fontweight="bold", y=1.002
)

for j, t in enumerate(["Original", "Detected mask", "Labeled domains", "Info"]):
    axes[0, j].set_title(t, fontsize=10, fontweight="bold")

for i, row in excluded_df.iterrows():
    img_path = os.path.join(IMAGE_DIR, row["image_name"])
    img      = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img_norm = img.astype(np.float32) / 255.0
    h, w     = img.shape
    nm_eff   = np.sqrt((DOMAIN_SIZE_NM / w) * (DOMAIN_SIZE_NM / h))

    thr  = filters.threshold_otsu(img_norm)
    mask = img_norm < thr
    mask = ndi.binary_fill_holes(mask)
    mask = morphology.remove_small_objects(mask, min_size=MIN_AREA)

    labels  = measure.label(mask)
    diam    = [p.equivalent_diameter * nm_eff for p in measure.regionprops(labels) if p.area >= MIN_AREA]
    bounds  = segmentation.find_boundaries(mask, mode="inner")
    overlay = np.dstack([img_norm, img_norm, img_norm])
    overlay[bounds] = [1, 0, 0]
    color   = reason_colors[row["reason"]]

    axes[i, 0].imshow(img, cmap="gray")
    for sp in axes[i, 0].spines.values():
        sp.set_edgecolor(color); sp.set_linewidth(3)
    axes[i, 0].set_ylabel(f"κ={row['kappa_gt']}  M={row['mob_gt']}", fontsize=9)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(overlay); axes[i, 1].axis("off")

    axes[i, 2].imshow(labels, cmap="nipy_spectral")
    axes[i, 2].set_xlabel(f"domains={len(diam)}", fontsize=8)
    axes[i, 2].axis("off")

    avg_d = np.mean(diam) if diam else float("nan")
    axes[i, 3].axis("off")
    axes[i, 3].text(
        0.05, 0.97,
        f"{row['image_name']}\n\nκ={row['kappa_gt']}  M={row['mob_gt']}\n"
        f"phase_frac:  {row['phase_fraction']:.4f}\n"
        f"avg_domain:  {avg_d:.2f} nm\n"
        f"domains:     {len(diam)}",
        transform=axes[i, 3].transAxes, fontsize=8, va="top", family="monospace"
    )
    axes[i, 3].text(
        0.05, 0.22, f"EXCLUDED:\n{row['reason']}",
        transform=axes[i, 3].transAxes, fontsize=9, va="top",
        color=color, fontweight="bold"
    )

plt.tight_layout()
save_path = os.path.join(OUT_DIR, "excluded_images_confirmation.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print("Saved:", save_path)

## Cosine Similarity
Computes pairwise cosine similarity between all clean images resized to 256×256.

`max_similarity` = highest similarity score to any other image in the dataset.
Used in plot (c) to show that visually similar microstructures lead to higher prediction error.

In [ ]:
# ================================================================
# CELL 8 — Cosine similarity
# Computed only on test images that have CNN predictions.
# out_df_test is used for panel (c) — error vs similarity.
# out_df_clean is kept for panels (a) and (b) — all clean images.
# ================================================================
 
from sklearn.metrics.pairwise import cosine_similarity
 
# filter to test images only — these are the ones with err_total values
out_df_test = out_df_clean[
    out_df_clean["err_total"].notna()
].copy().reset_index(drop=True)
 
print(f"Clean images total:           {len(out_df_clean)}")
print(f"Test images with predictions: {len(out_df_test)}")
 
SIM_SIZE    = (256, 256)
all_vectors = []
valid_idx   = []
 
for idx, img_name in enumerate(out_df_test["image_name"]):
    path = os.path.join(IMAGE_DIR, img_name)
    img  = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Missing: {img_name}")
        continue
    img_r = cv2.resize(img, SIM_SIZE, interpolation=cv2.INTER_AREA)
    all_vectors.append(img_r.astype(np.float32).flatten() / 255.0)
    valid_idx.append(idx)
 
all_vectors = np.vstack(all_vectors)
sim_matrix  = cosine_similarity(all_vectors)
np.fill_diagonal(sim_matrix, -1)
max_sim = np.max(sim_matrix, axis=1)
 
out_df_test["max_similarity"] = np.nan
out_df_test.loc[valid_idx, "max_similarity"] = max_sim
 
print(f"Similarity computed for: {len(valid_idx)} images")
print(f"Similarity range: {np.nanmin(max_sim):.4f} to {np.nanmax(max_sim):.4f}")

## Morphology Interpretability Plots
Produces the 3-panel morphology interpretability figure.

| Panel | x-axis | y-axis | color | Physical meaning |
|---|---|---|---|---|
| (a) | κ | Interface density | M | Higher κ → lower interface density |
| (b) | M | Avg domain size | κ | Higher M → more coarsening at fixed time |
| (c) | Max cosine similarity | Total error | M | Similar images → harder inverse problem |

Panel (d) is left empty — reserved for high-similarity microstructure examples (added in PowerPoint).

Saves to `fig6_morphology_analysis.png` at 600 DPI.

In [ ]:
# ================================================================
# CELL 9 — Final three plots (paper style, 2x2 layout)
# (a) and (b) use out_df_clean
# (c) uses out_df_test, test images with predictions only
# (d) empty — reserved for microstructure examples
# ================================================================
 
# ----- figure controls -----
FIG_WIDTH         = 13.0
FIG_HEIGHT        = 10.5
MAIN_HSPACE       = 0.30
MAIN_WSPACE       = 0.25
COLORBAR_PAD      = 0.01
COLORBAR_FRACTION = 0.05
FONT_FAMILY       = "Arial"
LABEL_SIZE        = 14
TICK_SIZE         = 11
LEGEND_SIZE       = 11
MARKER_SIZE       = 60
SAVE_DPI          = 600
 
plt.rcParams["font.family"]      = FONT_FAMILY
plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["axes.labelweight"] = "normal"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["xtick.labelsize"]  = TICK_SIZE
plt.rcParams["ytick.labelsize"]  = TICK_SIZE
 
KAPPA_SYMBOL = r"$\kappa$"
M_SYMBOL     = r"$M$"
 
 
def style_axis(ax):
    ax.tick_params(axis="both", labelsize=TICK_SIZE)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("normal")
    ax.grid(False)
 
 
def set_labels(ax, xlabel=None, ylabel=None):
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=LABEL_SIZE, fontweight="normal")
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=LABEL_SIZE, fontweight="normal")
 
 
fig = plt.figure(figsize=(FIG_WIDTH, FIG_HEIGHT))
gs  = GridSpec(2, 2, figure=fig, hspace=MAIN_HSPACE, wspace=MAIN_WSPACE)
 
ax00 = fig.add_subplot(gs[0, 0])
ax01 = fig.add_subplot(gs[0, 1])
ax10 = fig.add_subplot(gs[1, 0])
ax11 = fig.add_subplot(gs[1, 1])
 
 
# -----------------------------------
# (a) Interface density vs κ
# uses all clean images for full parameter space coverage
# -----------------------------------
sc0 = ax00.scatter(
    out_df_clean["kappa_gt"],
    out_df_clean["interface_density_nm_inv"],
    c=out_df_clean["mob_gt"],
    s=MARKER_SIZE,
    edgecolor="k",
    linewidth=0.4
)
set_labels(
    ax00,
    xlabel=rf"Ground truth {KAPPA_SYMBOL} ($\times10^{{-16}}$)",
    ylabel=r"Interface density (nm$^{-1}$)"
)
cb0 = fig.colorbar(sc0, ax=ax00, pad=COLORBAR_PAD, fraction=COLORBAR_FRACTION)
cb0.ax.set_title(
    r"$M$" + " " + r"($\times10^{-26}$)",
    fontsize=LEGEND_SIZE - 1, fontweight="normal", pad=6
)
cb0.ax.tick_params(labelsize=TICK_SIZE)
style_axis(ax00)
 
 
# -----------------------------------
# (b) Domain size vs M
# uses all clean images for full parameter space coverage
# -----------------------------------
sc1 = ax01.scatter(
    out_df_clean["mob_gt"],
    out_df_clean["avg_domain_size_nm"],
    c=out_df_clean["kappa_gt"],
    s=MARKER_SIZE,
    edgecolor="k",
    linewidth=0.4
)
set_labels(
    ax01,
    xlabel=rf"Ground truth {M_SYMBOL} ($\times10^{{-26}}$)",
    ylabel="Average domain size (nm)"
)
cb1 = fig.colorbar(sc1, ax=ax01, pad=COLORBAR_PAD, fraction=COLORBAR_FRACTION)
cb1.ax.set_title(
    r"$\kappa$" + " " + r"($\times10^{-16}$)",
    fontsize=LEGEND_SIZE - 1, fontweight="normal", pad=6
)
cb1.ax.tick_params(labelsize=TICK_SIZE)
style_axis(ax01)
 
 
# -----------------------------------
# (c) Error vs similarity
# uses test images only — only these have CNN prediction errors
# -----------------------------------
sc2 = ax10.scatter(
    out_df_test["max_similarity"],
    out_df_test["err_total"],
    c=out_df_test["mob_gt"],
    s=MARKER_SIZE,
    edgecolor="k",
    linewidth=0.4
)
set_labels(
    ax10,
    xlabel="Maximum cosine similarity",
    ylabel=r"MAE$_\kappa$ + MAE$_M$"
)
cb2 = fig.colorbar(sc2, ax=ax10, pad=COLORBAR_PAD, fraction=COLORBAR_FRACTION)
cb2.ax.set_title(
    r"$M$" + " " + r"($\times10^{-26}$)",
    fontsize=LEGEND_SIZE - 1, fontweight="normal", pad=6
)
cb2.ax.tick_params(labelsize=TICK_SIZE)
style_axis(ax10)
 
 
# -----------------------------------
# (d) Empty — reserved for microstructure images in PowerPoint
# -----------------------------------
ax11.axis("off")
 
 
# =====================================================
# Save
# =====================================================
 
save_path = os.path.join(OUT_DIR, "fig6_morphology_analysis.png")
plt.savefig(
    save_path,
    dpi=SAVE_DPI,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()
print("Saved:", save_path)

---
## Model Performance Figure

Produces the 4-panel model evaluation figure.

| Panel | Plot | Description |
|---|---|---|
| (a) top | R² vs layer index | κ and M accuracy across EfficientNetB7 feature layers |
| (a) bottom | MAE vs layer index | Combined MAE (κ + M) across layers |
| (b) | Training history | Train vs validation loss over epochs (log y-scale) |
| (c) | κ parity | Ground truth vs predicted κ, colored by M |
| (d) | M parity | Ground truth vs predicted M, colored by κ |

 **Paths below are independent of Cell 2 — edit here if files are in different locations.**

 **Key controls:**
- `SELECTED_LAYER` — highlights chosen layer with circle and dotted line
- `LAYER_XLIM / R2_YLIM / MAE_YLIM` — zoom ranges for layer sweep panels
- `LOSS_LOG_Y` — log scale on loss plot
- `MAKE_PARITY_SQUARE` — equal x/y range on parity plots
- `SHOW_TITLES` — toggle panel title labels on/off
- `SHOW_GRID` — toggle gridlines


---

In [ ]:
# ----- file paths -----
LAYER_CSV  = "/home/asfandyarkhan/Desktop/Papers/CNN_Paper/Our_Model/Results/CNN_mean_0.3766_range_0.02_Moose_Model/layer_sweep_EfficientNetB7/layer_sweep_results_B7.csv"
LOSS_CSV   = "/home/asfandyarkhan/Desktop/Papers/CNN_Paper/Our_Model/Results/400_set_layer464/loss_history_layer464.csv"
PRED_CSV   = "/home/asfandyarkhan/Desktop/Papers/CNN_Paper/Our_Model/Results/400_set_layer464/prediction_errors.csv"
FIG5_DIR   = "/home/asfandyarkhan/Desktop/Papers/CNN_Paper/Our_Model/Results/400_set_layer464/Paper_Figures"
os.makedirs(FIG5_DIR, exist_ok=True)

layer_df = pd.read_csv(LAYER_CSV)
loss_df  = pd.read_csv(LOSS_CSV)
pred_df  = pd.read_csv(PRED_CSV)

def find_col(df, possible_names):
    cols_lower = {c.lower().strip(): c for c in df.columns}
    for name in possible_names:
        if name.lower().strip() in cols_lower:
            return cols_lower[name.lower().strip()]
    raise ValueError(f"Could not find any of: {possible_names}\nAvailable: {list(df.columns)}")

# layer sweep columns
layer_col = find_col(layer_df, ["layer", "layer_index", "feature_extraction_layer", "layer_idx"])
r2_k_col  = find_col(layer_df, ["r2_kappa", "kappa_r2", "r2_k", "R2_kappa", "R2_k"])
r2_m_col  = find_col(layer_df, ["r2_mob",   "mob_r2",   "r2_M", "R2_mob",   "R2_M", "r2_m"])

try:
    mae_sum_col = find_col(layer_df, ["mae_sum", "combined_mae", "mae_total"])
    layer_df["MAE_sum"] = layer_df[mae_sum_col]
except Exception:
    mae_k_col = find_col(layer_df, ["mae_kappa", "kappa_mae", "mae_k", "MAE_kappa", "MAE_k"])
    mae_m_col = find_col(layer_df, ["mae_mob",   "mob_mae",   "mae_M", "MAE_mob",   "MAE_M", "mae_m"])
    layer_df["MAE_sum"] = layer_df[mae_k_col] + layer_df[mae_m_col]

# loss history columns
epoch_col    = find_col(loss_df, ["epoch", "epochs"])
loss_col     = find_col(loss_df, ["loss", "train_loss", "training_loss"])
val_loss_col = find_col(loss_df, ["val_loss", "validation_loss"])

# prediction columns
k_gt_col   = find_col(pred_df, ["kappa_gt",  "kappa_true",  "true_kappa",     "kappa_actual"])
k_pred_col = find_col(pred_df, ["kappa_pred","pred_kappa",  "predicted_kappa"])
m_gt_col   = find_col(pred_df, ["mob_gt",    "M_gt",        "mobility_gt",    "mob_true",    "M_true"])
m_pred_col = find_col(pred_df, ["mob_pred",  "M_pred",      "mobility_pred",  "pred_mobility","pred_M"])

k_true = pred_df[k_gt_col].values;   k_pred = pred_df[k_pred_col].values
m_true = pred_df[m_gt_col].values;   m_pred = pred_df[m_pred_col].values
r2_k   = r2_score(k_true, k_pred);   mae_k  = mean_absolute_error(k_true, k_pred)
r2_m   = r2_score(m_true, m_pred);   mae_m  = mean_absolute_error(m_true, m_pred)

# ----- figure controls -----
FIG_WIDTH        = 13.0;    FIG_HEIGHT    = 10.5
MAIN_HSPACE      = 0.30;    MAIN_WSPACE   = 0.25
LAYER_HSPACE     = 0.30
HEIGHT_RATIOS    = [1.0, 1.15];  WIDTH_RATIOS = [1.0, 1.0]
LAYER_XLIM       = (200, 800)
R2_YLIM          = (0.965, 1.0)
MAE_YLIM         = (0.27, 0.60)
LOSS_XLIM        = None;    LOSS_YLIM     = None
FONT_FAMILY      = "Arial"
TITLE_SIZE       = 13;      LABEL_SIZE    = 14
TICK_SIZE        = 11;      LEGEND_SIZE   = 11;  TEXTBOX_SIZE = 11
TITLE_WEIGHT     = "bold";  LABEL_WEIGHT  = "normal";  TICK_WEIGHT = "normal"
SHOW_TITLES      = False
MARKER_SIZE      = 35;      HIGHLIGHT_SIZE = 120;  LINE_WIDTH = 1.4
SHOW_GRID        = False;   GRID_ALPHA    = 0.25
LOSS_LOG_Y       = True
ADD_SELECTED_LAYER_TICK = True
SELECTED_LAYER   = 464
COLOR_KAPPA_LAYER = "blue";  COLOR_M_LAYER = "red";  COLOR_MAE_LAYER = "green"
COLOR_TRAIN_LOSS  = "red";   COLOR_VAL_LOSS = "blue"
KAPPA_SYMBOL     = r"$\kappa$";  KAPPA_HAT_SYMBOL = r"$\hat{\kappa}$"
MAKE_PARITY_SQUARE = False;  PARITY_PADDING = 0.3;  PARITY_MARKER_SIZE = 45
SAVE_DPI         = 600

plt.rcParams["font.family"]      = FONT_FAMILY
plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["axes.labelweight"] = LABEL_WEIGHT
plt.rcParams["axes.titleweight"] = TITLE_WEIGHT
plt.rcParams["xtick.labelsize"]  = TICK_SIZE
plt.rcParams["ytick.labelsize"]  = TICK_SIZE

# layout
fig = plt.figure(figsize=(FIG_WIDTH, FIG_HEIGHT))
gs  = GridSpec(2, 2, figure=fig, height_ratios=HEIGHT_RATIOS, width_ratios=WIDTH_RATIOS,
               hspace=MAIN_HSPACE, wspace=MAIN_WSPACE)
gs_layer = gs[0, 0].subgridspec(2, 1, hspace=LAYER_HSPACE)
ax_r2   = fig.add_subplot(gs_layer[0, 0])
ax_mae  = fig.add_subplot(gs_layer[1, 0], sharex=ax_r2)
ax_loss = fig.add_subplot(gs[0, 1])
ax_k    = fig.add_subplot(gs[1, 0])
ax_m    = fig.add_subplot(gs[1, 1])

def style_axis(ax):
    ax.tick_params(axis="both", labelsize=TICK_SIZE)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight(TICK_WEIGHT)
    ax.grid(True, alpha=GRID_ALPHA) if SHOW_GRID else ax.grid(False)

def set_title_if_needed(ax, title):
    if SHOW_TITLES:
        ax.set_title(title, fontsize=TITLE_SIZE, fontweight=TITLE_WEIGHT)

def set_labels(ax, xlabel=None, ylabel=None):
    if xlabel: ax.set_xlabel(xlabel, fontsize=LABEL_SIZE, fontweight=LABEL_WEIGHT)
    if ylabel: ax.set_ylabel(ylabel, fontsize=LABEL_SIZE, fontweight=LABEL_WEIGHT)

def apply_xlim_ylim(ax, xlim=None, ylim=None):
    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)

def make_parity_square(ax, tv, pv):
    lo = min(np.min(tv), np.min(pv)) - PARITY_PADDING
    hi = max(np.max(tv), np.max(pv)) + PARITY_PADDING
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_aspect("equal", adjustable="box")
    return lo, hi

def add_selected_layer_tick(ax, sel):
    ticks = sorted(set(list(ax.get_xticks()) + [sel]))
    ax.set_xticks(ticks)
    labels = [str(int(t)) for t in ticks]
    ticklabels = ax.set_xticklabels(labels, fontsize=TICK_SIZE)
    for lbl in ticklabels:
        if lbl.get_text() == str(sel):
            lbl.set_fontsize(TICK_SIZE - 3); lbl.set_fontweight("bold")

# (a) layer sweep — R2
ax_r2.scatter(layer_df[layer_col], layer_df[r2_k_col], label=KAPPA_SYMBOL, s=MARKER_SIZE, color=COLOR_KAPPA_LAYER)
ax_r2.scatter(layer_df[layer_col], layer_df[r2_m_col], label=r"$M$",       s=MARKER_SIZE, color=COLOR_M_LAYER)

if SELECTED_LAYER in layer_df[layer_col].values:
    row_sel = layer_df[layer_df[layer_col] == SELECTED_LAYER].iloc[0]
    mid_r2  = (row_sel[r2_k_col] + row_sel[r2_m_col]) / 2
    ax_r2.scatter(SELECTED_LAYER, mid_r2, s=HIGHLIGHT_SIZE*2.7, facecolors="none", edgecolors="k", linewidth=1.5)
    ax_mae.scatter(SELECTED_LAYER, row_sel["MAE_sum"], s=HIGHLIGHT_SIZE, facecolors="none", edgecolors="k", linewidth=1.5)
    ax_r2.plot([SELECTED_LAYER, SELECTED_LAYER], [R2_YLIM[0],  mid_r2],            linestyle=":", linewidth=1, color="k")
    ax_mae.plot([SELECTED_LAYER, SELECTED_LAYER], [MAE_YLIM[0], row_sel["MAE_sum"]], linestyle=":", linewidth=1, color="k")

set_title_if_needed(ax_r2, "(a) Feature extraction layer selection")
set_labels(ax_r2, ylabel=r"$R^2$")
ax_r2.legend(frameon=True, fontsize=LEGEND_SIZE, loc="lower right", bbox_to_anchor=(0.65, 0.1), edgecolor="black")
apply_xlim_ylim(ax_r2, xlim=LAYER_XLIM, ylim=R2_YLIM)
ax_r2.set_yticks([0.97, 0.98, 0.99, 1.00])
ax_r2.set_yticklabels(["0.97", "0.98", "0.99", "1.00"])
style_axis(ax_r2)

# (a) layer sweep — MAE
ax_mae.scatter(layer_df[layer_col], layer_df["MAE_sum"], s=MARKER_SIZE, color=COLOR_MAE_LAYER)
set_labels(ax_mae, xlabel="Feature extraction layer index", ylabel=r"MAE$_\kappa$ + MAE$_M$")
apply_xlim_ylim(ax_mae, xlim=LAYER_XLIM, ylim=MAE_YLIM)
style_axis(ax_mae)

if ADD_SELECTED_LAYER_TICK:
    add_selected_layer_tick(ax_r2,  SELECTED_LAYER)
    add_selected_layer_tick(ax_mae, SELECTED_LAYER)

# (b) training history
ax_loss.plot(loss_df[epoch_col], loss_df[loss_col],     label="Train loss",      linewidth=LINE_WIDTH, color=COLOR_TRAIN_LOSS)
ax_loss.plot(loss_df[epoch_col], loss_df[val_loss_col], label="Validation loss", linewidth=LINE_WIDTH, color=COLOR_VAL_LOSS)

best_epoch = loss_df.loc[loss_df[val_loss_col].idxmin(), epoch_col]
best_val   = loss_df[val_loss_col].min()
ax_loss.plot([best_epoch, best_epoch], [ax_loss.get_ylim()[0], best_val],
             linestyle=":", linewidth=1.2, color="k", label=f"Best epoch ({int(best_epoch)})")

if LOSS_LOG_Y: ax_loss.set_yscale("log")
set_title_if_needed(ax_loss, "(b) Training history")
set_labels(ax_loss, xlabel="Epoch", ylabel="Loss")
ax_loss.legend(frameon=True, fontsize=LEGEND_SIZE, loc="upper right", edgecolor="black")
apply_xlim_ylim(ax_loss, xlim=LOSS_XLIM, ylim=LOSS_YLIM)
style_axis(ax_loss)

# (c) κ parity
sc_k = ax_k.scatter(k_true, k_pred, c=m_true, s=PARITY_MARKER_SIZE, edgecolor="k", linewidth=0.4)
if MAKE_PARITY_SQUARE:
    mn_k, mx_k = make_parity_square(ax_k, k_true, k_pred)
else:
    mn_k = min(np.min(k_true), np.min(k_pred))
    mx_k = max(np.max(k_true), np.max(k_pred))
ax_k.plot([mn_k, mx_k], [mn_k, mx_k], "k-", linewidth=LINE_WIDTH)
set_title_if_needed(ax_k, rf"(c) Gradient-energy coefficient ({KAPPA_SYMBOL})")
set_labels(ax_k,
    xlabel=rf"Ground truth {KAPPA_SYMBOL} ($\times10^{{-16}}$)",
    ylabel=rf"Prediction {KAPPA_HAT_SYMBOL} ($\times10^{{-16}}$)")
ax_k.text(0.06, 0.88, f"$R^2$ = {r2_k:.3f}\nMAE = {mae_k:.3f}" + r" $\times10^{-16}$",
          transform=ax_k.transAxes, fontsize=TEXTBOX_SIZE, fontfamily="Arial",
          bbox=dict(facecolor="white", edgecolor="black", alpha=0.9))
style_axis(ax_k)
cb_k = fig.colorbar(sc_k, ax=ax_k, pad=0.02, fraction=0.05)
cb_k.ax.set_title(r"$M$" + " " + r"($\times10^{-26}$)", fontsize=LEGEND_SIZE-1, fontweight="normal", pad=6)
cb_k.ax.tick_params(labelsize=TICK_SIZE)

# (d) M parity
sc_m = ax_m.scatter(m_true, m_pred, c=k_true, s=PARITY_MARKER_SIZE, edgecolor="k", linewidth=0.4)
if MAKE_PARITY_SQUARE:
    mn_m, mx_m = make_parity_square(ax_m, m_true, m_pred)
else:
    mn_m = min(np.min(m_true), np.min(m_pred))
    mx_m = max(np.max(m_true), np.max(m_pred))
ax_m.plot([mn_m, mx_m], [mn_m, mx_m], "k-", linewidth=LINE_WIDTH)
set_title_if_needed(ax_m, r"(d) Effective mobility ($M$)")
set_labels(ax_m,
    xlabel=r"Ground truth $M$ ($\times10^{-26}$)",
    ylabel=r"Prediction $\hat{M}$ ($\times10^{-26}$)")
ax_m.text(0.06, 0.88, f"$R^2$ = {r2_m:.3f}\nMAE = {mae_m:.3f}" + r" $\times10^{-26}$",
          transform=ax_m.transAxes, fontsize=TEXTBOX_SIZE, fontfamily="Arial",
          bbox=dict(facecolor="white", edgecolor="black", alpha=0.9))
style_axis(ax_m)
cb_m = fig.colorbar(sc_m, ax=ax_m, pad=0.02, fraction=0.05)
cb_m.ax.set_title(r"$\kappa$" + " " + r"($\times10^{-16}$)", fontsize=LEGEND_SIZE-1, fontweight="normal", pad=6)
cb_m.ax.tick_params(labelsize=TICK_SIZE)

save_path = os.path.join(OUT_DIR, "figure_model_performance_combined.png")
plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved:", save_path)

---
## Interpolation and Extended Grid Parity

Produces the 2×2 parity figure comparing two independent test sets.

| Row | Panels | Dataset | Purpose |
|---|---|---|---|
| Top | (a) κ, (b) M | `interpolation_test_results.csv` | Off-grid test — (κ, M) combinations between training grid points |
| Bottom | (c) κ, (d) M | `prediction_errors.csv` (760-set) | Extended mobility range — M up to 100 × 10⁻²⁶ |

 `MAKE_PARITY_SQUARE = True` forces equal axes. Saves to `figure_parity_plots_combined.png`.

---

In [ ]:
# ----- file paths -----
INTERP_CSV = "/home/asfandyarkhan/Desktop/Papers/CNN_Paper/Our_Model/Results/400_set_layer464/interpolation_test_results.csv"
PRED_CSV   = "/home/asfandyarkhan/Desktop/Papers/CNN_Paper/Our_Model/Results/760_set_layer464/prediction_errors.csv"

interp_df = pd.read_csv(INTERP_CSV)
pred_df   = pd.read_csv(PRED_CSV)

# column detection — handles minor naming differences between CSV files
def find_col(df, possible_names):
    cols_lower = {c.lower().strip(): c for c in df.columns}
    for name in possible_names:
        if name.lower().strip() in cols_lower:
            return cols_lower[name.lower().strip()]
    raise ValueError(f"Could not find any of: {possible_names}\nAvailable: {list(df.columns)}")

ik_gt_col   = find_col(interp_df, ["kappa_true", "kappa_gt",  "true_kappa",    "kappa_actual"])
ik_pred_col = find_col(interp_df, ["kappa_pred", "pred_kappa","predicted_kappa"])
im_gt_col   = find_col(interp_df, ["mob_true",   "mob_gt",    "M_gt",          "true_mobility","M_true"])
im_pred_col = find_col(interp_df, ["mob_pred",   "M_pred",    "mobility_pred", "pred_mobility","pred_M"])

k_gt_col    = find_col(pred_df,   ["kappa_gt",   "kappa_true","true_kappa",    "kappa_actual"])
k_pred_col  = find_col(pred_df,   ["kappa_pred", "pred_kappa","predicted_kappa"])
m_gt_col    = find_col(pred_df,   ["mob_gt",     "M_gt",      "mobility_gt",   "mob_true",    "M_true"])
m_pred_col  = find_col(pred_df,   ["mob_pred",   "M_pred",    "mobility_pred", "pred_mobility","pred_M"])

ik_true = interp_df[ik_gt_col].values;  ik_pred = interp_df[ik_pred_col].values
im_true = interp_df[im_gt_col].values;  im_pred = interp_df[im_pred_col].values
r2_ik   = r2_score(ik_true, ik_pred);   mae_ik  = mean_absolute_error(ik_true, ik_pred)
r2_im   = r2_score(im_true, im_pred);   mae_im  = mean_absolute_error(im_true, im_pred)

k_true  = pred_df[k_gt_col].values;     k_pred  = pred_df[k_pred_col].values
m_true  = pred_df[m_gt_col].values;     m_pred  = pred_df[m_pred_col].values
r2_k    = r2_score(k_true, k_pred);     mae_k   = mean_absolute_error(k_true, k_pred)
r2_m    = r2_score(m_true, m_pred);     mae_m   = mean_absolute_error(m_true, m_pred)

# ----- figure controls -----
FIG_WIDTH  = 13.0;  FIG_HEIGHT = 10.5
MAIN_WSPACE = 0.25; MAIN_HSPACE = 0.30
COLORBAR_PAD = 0.02; COLORBAR_FRACTION = 0.05
FONT_FAMILY = "Arial"
LABEL_SIZE = 14;  TICK_SIZE = 11;  LEGEND_SIZE = 11;  TEXTBOX_SIZE = 11
LABEL_WEIGHT = "normal";  TICK_WEIGHT = "normal"
MARKER_SIZE = 45;  LINE_WIDTH = 1.4
MAKE_PARITY_SQUARE = False;  PARITY_PADDING = 0.3
KAPPA_SYMBOL = r"$\kappa$";  KAPPA_HAT_SYMBOL = r"$\hat{\kappa}$"
SAVE_DPI = 600

plt.rcParams["font.family"]      = FONT_FAMILY
plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["axes.labelweight"] = LABEL_WEIGHT
plt.rcParams["xtick.labelsize"]  = TICK_SIZE
plt.rcParams["ytick.labelsize"]  = TICK_SIZE

def style_axis(ax):
    ax.tick_params(axis="both", labelsize=TICK_SIZE)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight(TICK_WEIGHT)
    ax.grid(False)

def set_labels(ax, xlabel=None, ylabel=None):
    if xlabel: ax.set_xlabel(xlabel, fontsize=LABEL_SIZE, fontweight=LABEL_WEIGHT)
    if ylabel: ax.set_ylabel(ylabel, fontsize=LABEL_SIZE, fontweight=LABEL_WEIGHT)

def parity_panel(ax, fig, x_true, x_pred, color_vals, xlabel, ylabel, textbox_str, cb_title):
    sc = ax.scatter(x_true, x_pred, c=color_vals, s=MARKER_SIZE, edgecolor="k", linewidth=0.4)
    mn = min(np.min(x_true), np.min(x_pred))
    mx = max(np.max(x_true), np.max(x_pred))
    ax.plot([mn, mx], [mn, mx], "k-", linewidth=LINE_WIDTH)
    if MAKE_PARITY_SQUARE:
        pad = PARITY_PADDING * (mx - mn)
        ax.set_xlim(mn - pad, mx + pad); ax.set_ylim(mn - pad, mx + pad)
        ax.set_aspect("equal")
    set_labels(ax, xlabel=xlabel, ylabel=ylabel)
    ax.text(0.06, 0.88, textbox_str, transform=ax.transAxes, fontsize=TEXTBOX_SIZE,
            fontfamily="Arial", bbox=dict(facecolor="white", edgecolor="black", alpha=0.9))
    style_axis(ax)
    cb = fig.colorbar(sc, ax=ax, pad=COLORBAR_PAD, fraction=COLORBAR_FRACTION)
    cb.ax.set_title(cb_title, fontsize=LEGEND_SIZE-1, fontweight="normal", pad=6)
    cb.ax.tick_params(labelsize=TICK_SIZE)
    return sc

fig, axes = plt.subplots(2, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))
fig.subplots_adjust(wspace=MAIN_WSPACE, hspace=MAIN_HSPACE)
ax_a, ax_b = axes[0]
ax_c, ax_d = axes[1]

# row 1 — interpolation test
parity_panel(ax_a, fig, ik_true, ik_pred, im_true,
    xlabel=rf"Ground truth {KAPPA_SYMBOL} ($\times10^{{-16}}$)",
    ylabel=rf"Prediction {KAPPA_HAT_SYMBOL} ($\times10^{{-16}}$)",
    textbox_str=f"$R^2$ = {r2_ik:.3f}\nMAE = {mae_ik:.3f}" + r" $\times10^{-16}$",
    cb_title=r"$M$" + " " + r"($\times10^{-26}$)")

parity_panel(ax_b, fig, im_true, im_pred, ik_true,
    xlabel=r"Ground truth $M$ ($\times10^{-26}$)",
    ylabel=r"Prediction $\hat{M}$ ($\times10^{-26}$)",
    textbox_str=f"$R^2$ = {r2_im:.3f}\nMAE = {mae_im:.3f}" + r" $\times10^{-26}$",
    cb_title=r"$\kappa$" + " " + r"($\times10^{-16}$)")

# row 2 — extended grid test
parity_panel(ax_c, fig, k_true, k_pred, m_true,
    xlabel=rf"Ground truth {KAPPA_SYMBOL} ($\times10^{{-16}}$)",
    ylabel=rf"Prediction {KAPPA_HAT_SYMBOL} ($\times10^{{-16}}$)",
    textbox_str=f"$R^2$ = {r2_k:.3f}\nMAE = {mae_k:.3f}" + r" $\times10^{-16}$",
    cb_title=r"$M$" + " " + r"($\times10^{-26}$)")

parity_panel(ax_d, fig, m_true, m_pred, k_true,
    xlabel=r"Ground truth $M$ ($\times10^{-26}$)",
    ylabel=r"Prediction $\hat{M}$ ($\times10^{-26}$)",
    textbox_str=f"$R^2$ = {r2_m:.3f}\nMAE = {mae_m:.3f}" + r" $\times10^{-26}$",
    cb_title=r"$\kappa$" + " " + r"($\times10^{-16}$)")

save_path = os.path.join(OUT_DIR, "figure_parity_plots_combined.png")
plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved:", save_path)

---
## Experimental Image Preprocessing (STEM Pipeline)

Preprocesses the experimental HAADF-STEM image from Westraadt et al. [32] for CNN inference.

**Input:** Experimental microstructure from Figure 1a (Fe–36 wt.% Cr, 500 °C, 100 h).

| Step | Output file | Description |
|---|---|---|
| 0 | `_0_gray.png` | Grayscale conversion |
| 1 | `_1_denoise.png` | Median + bilateral denoising |
| 2 | `_2_enhanced.png` | Background normalization (large-sigma Gaussian subtraction) |
| 3 | `_3_mask_raw.png` | Otsu threshold + morphological cleanup |
| 4 | `_4_mask_smooth.png` | Boundary smoothing |
| 5 | `_5_whitebg_raw.png` | Grayscale Cr-rich regions on white (raw mask) |
| 6 | `_6_whitebg_smooth.png` | **← CNN INPUT** (smoothed mask version) |

 **Key parameters to tune if output looks wrong:**
- `bg_sigma` — background normalization strength (~10–15% of image width)
- `smooth_sigma` + `smooth_thr` — domain roundness and boundary growth
- `threshold_mode` — switch to `"manual"` if Otsu picks the wrong class

---

In [ ]:
STEM_CFG = {
    "img_path": (
        "/home/asfandyarkhan/Desktop/Papers/CNN_Paper/Our_Model/"
        "Images/Experimental_Images/J.E_Westraadt_2015/"
        "Figure_1a_cleanedChatGpt.png"
    ),
    "save_prefix": (
        "/home/asfandyarkhan/Desktop/Papers/CNN_Paper/Our_Model/"
        "Images/Experimental_Images/J.E_Westraadt_2015/processed/stem_panelA"
    ),
    "median_ksize":          3,
    "bilateral_d":           7,
    "bilateral_sigma_color": 35,
    "bilateral_sigma_space": 35,
    "bg_sigma":              120,   # large-sigma Gaussian for background normalization
    "threshold_mode":        "otsu",
    "manual_thr":            25,
    "open_ksize":            20,    # morphological opening to remove noise before smoothing
    "close_ksize":            7,
    "min_area":              30,
    "smooth_sigma":          12.0,  # higher = rounder domain boundaries
    "smooth_thr":            12,    # lower = domains grow outward
    "smooth_close_ksize":    15,
    "smooth_open_ksize":      0,    # 0 = disabled
    "show_steps":            True,
}


def show_img(title, img, cmap="gray", figsize=(5, 5)):
    plt.figure(figsize=figsize)
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap)
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def smooth_stem_mask(mask, smooth_sigma, smooth_thr, close_ksize, open_ksize):
    # Gaussian blur softens binary mask edges, then re-threshold gives rounded boundaries
    soft = cv2.GaussianBlur(mask, (0, 0), smooth_sigma)
    _, sm = cv2.threshold(soft, smooth_thr, 255, cv2.THRESH_BINARY)
    if close_ksize and close_ksize > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_ksize, close_ksize))
        sm = cv2.morphologyEx(sm, cv2.MORPH_CLOSE, k)
    if open_ksize and open_ksize > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_ksize, open_ksize))
        sm = cv2.morphologyEx(sm, cv2.MORPH_OPEN, k)
    return sm


def run_stem_cleanup(cfg=STEM_CFG):

    os.makedirs(os.path.dirname(cfg["save_prefix"]), exist_ok=True)

    bgr = cv2.imread(cfg["img_path"])
    if bgr is None:
        raise FileNotFoundError(f"Could not read: {cfg['img_path']}")

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    print(f"Image size: {w} x {h} px")

    # denoise
    den1 = cv2.medianBlur(gray, cfg["median_ksize"])
    den2 = cv2.bilateralFilter(den1, cfg["bilateral_d"],
                               cfg["bilateral_sigma_color"], cfg["bilateral_sigma_space"])

    # background normalization: Cr = dark, Fe = bright
    bg       = cv2.GaussianBlur(den2, (0, 0), cfg["bg_sigma"])
    enhanced = cv2.subtract(bg, den2)
    enh      = cv2.normalize(enhanced, None, 0, 255, cv2.NORM_MINMAX)

    # threshold
    if cfg["threshold_mode"].lower() == "otsu":
        _, mask0 = cv2.threshold(enh, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        _, mask0 = cv2.threshold(enh, int(cfg["manual_thr"]), 255, cv2.THRESH_BINARY)

    # morphological cleanup
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (cfg["open_ksize"],  cfg["open_ksize"]))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (cfg["close_ksize"], cfg["close_ksize"]))
    mask1 = cv2.morphologyEx(mask0, cv2.MORPH_OPEN,  k_open)
    mask2 = cv2.morphologyEx(mask1, cv2.MORPH_CLOSE, k_close)

    # remove tiny components
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask2, connectivity=8)
    mask = np.zeros_like(mask2)
    for i in range(1, num):
        if stats[i, cv2.CC_STAT_AREA] >= cfg["min_area"]:
            mask[labels == i] = 255

    # smooth boundaries for rounded domain shapes
    mask_s = smooth_stem_mask(mask,
                              smooth_sigma=cfg["smooth_sigma"],
                              smooth_thr=cfg["smooth_thr"],
                              close_ksize=cfg["smooth_close_ksize"],
                              open_ksize=cfg["smooth_open_ksize"])

    # build outputs — grayscale Cr-rich regions on white background
    # matches the continuous grayscale appearance of synthetic training images
    white_raw    = np.full_like(gray, 255);  white_raw[mask > 0]    = den2[mask > 0]
    white_smooth = np.full_like(gray, 255);  white_smooth[mask_s > 0] = den2[mask_s > 0]

    pf = (mask_s > 0).mean()
    print(f"Phase fraction (Cr-rich): {pf:.3f}  (target ~0.22–0.25)")

    pref = cfg["save_prefix"]
    cv2.imwrite(f"{pref}_0_gray.png",          gray)
    cv2.imwrite(f"{pref}_1_denoise.png",        den2)
    cv2.imwrite(f"{pref}_2_enhanced.png",       enh)
    cv2.imwrite(f"{pref}_3_mask_raw.png",       mask)
    cv2.imwrite(f"{pref}_4_mask_smooth.png",    mask_s)
    cv2.imwrite(f"{pref}_5_whitebg_raw.png",    white_raw)
    cv2.imwrite(f"{pref}_6_whitebg_smooth.png", white_smooth)

    if cfg["show_steps"]:
        show_img("0) Input (gray)",                     gray)
        show_img("1) Denoised",                         den2)
        show_img("2) Enhanced (bg − img)",              enh)
        show_img("3) Mask (raw)",                       mask)
        show_img(f"4) Mask (smoothed)  sigma={cfg['smooth_sigma']}  thr={cfg['smooth_thr']}", mask_s)
        show_img("5) White BG — raw mask",              white_raw)
        show_img("6) White BG — smoothed  ← CNN INPUT", white_smooth)

    print("\nSaved:")
    for s in ["_0_gray.png","_1_denoise.png","_2_enhanced.png","_3_mask_raw.png",
              "_4_mask_smooth.png","_5_whitebg_raw.png","_6_whitebg_smooth.png"]:
        print(f"  {pref}{s}")
    print(f"\n→ CNN input: {pref}_6_whitebg_smooth.png")

    return white_smooth, mask_s


cnn_input, stem_mask = run_stem_cleanup(STEM_CFG)